# Fine-Tuning Laya on Daily Market Predictor Data

This notebook fine-tunes **Laya** (`convaiinnovations/laya-typed-decisions`), a non-autoregressive System 1 decision engine built on `ModernBERT-large`, to predict intraday market direction (`choice: UP/DOWN`) over full pre-market macro contexts (~5,000 tokens).

### Workflow
1. **Load Pre-Market Data**: Loads `train.jsonl` and `val.jsonl` formatted in Laya's typed-decisions schema.
2. **LoRA Parameter Efficient Adaptation**: Attaches low-rank adapters ($r=16, \alpha=16$) to ModernBERT's attention projection layers (`Wqkv`, `Wo`).
3. **Training**: Minimizes binary cross-entropy on direction classification.
4. **Post-Hoc Temperature Calibration**: Fits temperature $T$ to align confidence with true empirical probabilities, minimizing Brier score.

In [ ]:
# Install dependencies (for Colab / Kaggle / Local)
!pip install -q torch transformers peft accelerate datasets python-dotenv
!pip install -q -U torchao
!mkdir -p data && mv train.jsonl val.jsonl data/ 2>/dev/null || true

In [ ]:
import json
from pathlib import Path

# Check sample data (checks both data/ and root)
train_path = Path('data/train.jsonl') if Path('data/train.jsonl').exists() else Path('train.jsonl')
val_path = Path('data/val.jsonl') if Path('data/val.jsonl').exists() else Path('val.jsonl')

if train_path.exists():
    with open(train_path, 'r') as f:
        sample = json.loads(f.readline())
        print(f"Using: {train_path}")
        print(f"Sample Date: {sample['metadata']['target_date']}")
        print(f"Direction Target: {sample['gold']['direction']['choice']}")
        print(f"Context Length: {len(sample['state'])} characters")
else:
    print('Please upload train.jsonl and val.jsonl to data/ or root')

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = 'answerdotai/ModernBERT-large'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {DEVICE}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or '[PAD]'

class LayaDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=8192):
        self.samples = []
        with open(path, 'r') as f:
            for line in f:
                if line.strip():
                    self.samples.append(json.loads(line))
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        q = item['questions']['direction']
        prompt = f"Instruction: {q['instructions']}\nOptions:\nUP: {q['options']['UP']}\nDOWN: {q['options']['DOWN']}\n\nMarket Context:\n{item['state']}"
        label = 1 if item['gold']['direction']['choice'] == 'UP' else 0
        enc = self.tokenizer(prompt, max_length=self.max_len, truncation=True, return_tensors=None)
        return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'], 'label': label}

def collate_fn(batch, pad_id=0):
    max_len = max(len(b['input_ids']) for b in batch)
    ids, mask, labels = [], [], []
    for b in batch:
        pad = max_len - len(b['input_ids'])
        ids.append(b['input_ids'] + [pad_id] * pad)
        mask.append(b['attention_mask'] + [0] * pad)
        labels.append(b['label'])
    return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(mask), 'labels': torch.tensor(labels)}

train_file = 'data/train.jsonl' if Path('data/train.jsonl').exists() else 'train.jsonl'
val_file = 'data/val.jsonl' if Path('data/val.jsonl').exists() else 'val.jsonl'
print(f"Loading train dataset from: {train_file}")
print(f"Loading val dataset from:   {val_file}")

train_loader = DataLoader(LayaDataset(train_file, tokenizer), batch_size=1, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(LayaDataset(val_file, tokenizer), batch_size=1, shuffle=False, collate_fn=collate_fn)

In [ ]:
# Free any residual GPU memory
import gc
gc.collect()
torch.cuda.empty_cache()

# Initialize Model with LoRA targeting ModernBERT layers in bfloat16 with gradient checkpointing
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'DOWN', 1: 'UP'},
    label2id={'DOWN': 0, 'UP': 1},
    dtype=torch.bfloat16
)

# Enable gradient checkpointing to allow larger batch sizes
base_model.gradient_checkpointing_enable()

# Target attention (Wqkv, Wo) and MLP projections (Wi, Wo)
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['Wqkv', 'Wo', 'Wi'],
    bias='none'
)
model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()
model.to(DEVICE)

In [ ]:
# Fine-Tuning Loop with Per-Epoch Validation & Checkpoint Restoration
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

EPOCHS = 20
BATCH_SIZE = 4
GRAD_ACCUM = 2

train_loader = DataLoader(
    LayaDataset(train_file, tokenizer),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.02)
total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.05 * total_steps),
    num_training_steps=total_steps,
)

# Label smoothing prevents overconfidence on noisy market labels
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

best_val_brier = float('inf')
best_epoch = 0
best_state = None

print(f"{'Epoch':>7} | {'Train Loss':>10} | {'Val Acc':>8} | {'Val Brier':>10} | Status")
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        logits = model(input_ids=ids, attention_mask=mask).logits
        loss = criterion(logits, labels) / GRAD_ACCUM
        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    avg_train_loss = total_loss / len(train_loader)

    # Validation pass
    model.eval()
    val_logits_list, val_labels_list = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            val_logits_list.append(model(input_ids=ids, attention_mask=mask).logits.float().cpu())
            val_labels_list.append(batch['labels'].cpu())

    val_logits = torch.cat(val_logits_list, dim=0)
    val_labels = torch.cat(val_labels_list, dim=0)
    val_p = torch.softmax(val_logits, dim=-1)[:, 1].numpy()
    val_y = val_labels.numpy()

    val_acc = float(((val_p >= 0.5) == val_y).mean())
    val_brier = float(((val_p - val_y) ** 2).mean())

    status = ''
    if val_brier < best_val_brier:
        best_val_brier = val_brier
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        status = '★ Best'

    print(f"Epoch {epoch:2d} | {avg_train_loss:10.4f} | {val_acc*100:7.1f}% | {val_brier:10.4f} | {status}")

if best_state is not None:
    print(f"\nRestoring best model checkpoint from Epoch {best_epoch} (Val Brier: {best_val_brier:.4f})")
    model.load_state_dict(best_state)


In [ ]:
# Temperature & Platt Scaling Calibration Fitting
from sklearn.linear_model import LogisticRegression

model.eval()
val_logits_list, val_labels_list = [], []
with torch.no_grad():
    for batch in val_loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        val_logits_list.append(model(input_ids=ids, attention_mask=mask).logits.float().cpu())
        val_labels_list.append(batch['labels'].cpu())

val_logits = torch.cat(val_logits_list, dim=0)
val_labels = torch.cat(val_labels_list, dim=0)

# 1. Scalar Temperature Scaler
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)
    def forward(self, x):
        return x / torch.clamp(self.temperature, 0.1, 5.0)

scaler = TemperatureScaler()
opt = torch.optim.LBFGS([scaler.temperature], lr=0.01, max_iter=100)
loss_fn = nn.CrossEntropyLoss()

def eval_temp():
    opt.zero_grad()
    loss_val = loss_fn(scaler(val_logits), val_labels)
    loss_val.backward()
    return loss_val

opt.step(eval_temp)
fitted_t = float(scaler.temperature.item())

raw_probs = torch.softmax(val_logits, dim=-1)[:, 1].numpy()
temp_probs = torch.softmax(scaler(val_logits), dim=-1)[:, 1].detach().numpy()
val_y = val_labels.numpy()

raw_brier = float(((raw_probs - val_y) ** 2).mean())
temp_brier = float(((temp_probs - val_y) ** 2).mean())
acc = float(((raw_probs >= 0.5) == val_y).mean())

# 2. Platt Scaling (Logistic Regression on logit difference: delta_z)
val_logits_np = (val_logits[:, 1] - val_logits[:, 0]).numpy().reshape(-1, 1)
lr_calibrator = LogisticRegression(C=1.0, solver='lbfgs')
lr_calibrator.fit(val_logits_np, val_y)

platt_probs = lr_calibrator.predict_proba(val_logits_np)[:, 1]
platt_brier = float(((platt_probs - val_y) ** 2).mean())
platt_acc = float(((platt_probs >= 0.5) == val_y).mean())
platt_w = float(lr_calibrator.coef_[0][0])
platt_b = float(lr_calibrator.intercept_[0])

print(f"Raw Accuracy:          {acc * 100:.1f}%")
print(f"Raw Brier Score:       {raw_brier:.4f}")
print(f"Fitted Temp (T):       {fitted_t:.3f}")
print(f"Temperature Brier:     {temp_brier:.4f}")
print(f"Platt Scaled Acc:      {platt_acc * 100:.1f}%")
print(f"Platt Calibrated Brier:{platt_brier:.4f} (w={platt_w:.4f}, b={platt_b:.4f})")


In [ ]:
# Export Fine-Tuned Model Weights, Tokenizer, and Calibration Config
import json
from pathlib import Path

output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# 1. Save LoRA adapter weights and tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# 2. Save calibration config
cal_config = {
    'base_model': MODEL_NAME,
    'method': 'platt_scaling',
    'platt_w': platt_w,
    'platt_b': platt_b,
    'fitted_temperature': fitted_t,
    'raw_accuracy': acc,
    'platt_accuracy': platt_acc,
    'raw_brier': raw_brier,
    'temperature_brier': temp_brier,
    'calibrated_brier': platt_brier,
    'best_epoch': best_epoch,
}

with open(output_dir / 'laya_calibration_config.json', 'w') as f:
    json.dump(cal_config, f, indent=2)

print(f"Artifacts successfully saved to {output_dir}/")
print(json.dumps(cal_config, indent=2))

# 3. Zip for 1-click download
!zip -r laya_weights.zip output/
